In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Optional, List
import time
import pandas as pd


# ---------- 1) Modelo do "tick" ----------
@dataclass(frozen=True)
class TelemetryTick:
    execution: int
    t: int
    h: float
    dt: float
    b: float
    armed: bool
    wind_speed: float
    humidity: float
    vibration: float
    action: Optional[str]  # None quando não houve ação naquele tick


# ---------- 2) Bus (publish/subscribe) ----------
class TelemetryBus:
    def __init__(self) -> None:
        self._subs: List[Callable[[TelemetryTick], None]] = []

    def subscribe(self, fn: Callable[[TelemetryTick], None]) -> None:
        self._subs.append(fn)

    def publish(self, tick: TelemetryTick) -> None:
        for fn in self._subs:
            fn(tick)


# ---------- 3) Monitor (consome ticks e guarda o "último conhecido") ----------
class RuntimeDataMonitor:
    def __init__(self) -> None:
        self.latest: Optional[TelemetryTick] = None
        self.history_runtime_data: List[TelemetryTick] = []  # opcional (pode desligar se ficar grande)

    def handle_tick(self, tick: TelemetryTick) -> None:
        self.latest = tick
        self.history_runtime_data.append(tick)

        # exemplo de log
        act = tick.action if tick.action else "-"
        print(f"[exec={tick.execution} t={tick.t:02d}] action={act:15s} h={tick.h:6.1f} b={tick.b:5.1f} vib={tick.vibration:.2f}")


# ---------- 4) Replayer (lê o CSV e solta tick a tick) ----------
def _clean_actions(s: pd.Series) -> pd.Series:
    # Action vem com espaços e NaN. Vamos normalizar:
    s = s.fillna("-").astype(str).str.strip()
    s = s.replace({"nan": "-", "None": "-"})
    return s

def load_ticks(csv_path: str, execution: int) -> List[TelemetryTick]:
    df = pd.read_csv(csv_path)

    # normaliza Action
    df["Action"] = _clean_actions(df["Action"])

    # garante ordenação
    df = df[df["execution"] == execution].sort_values(["execution", "t"], ascending=True)

    ticks: List[TelemetryTick] = []
    for _, r in df.iterrows():
        action = r["Action"]
        action = None if action == "-" else str(action)

        ticks.append(
            TelemetryTick(
                execution=int(r["execution"]),
                t=int(r["t"]),
                h=float(r["h"]),
                dt=float(r["dt"]),
                b=float(r["b"]),
                armed=bool(r["Armed_Status"]),
                wind_speed=float(r["Wind_Speed"]),
                humidity=float(r["Humidity"]),
                vibration=float(r["Vibration"]),
                action=action,
            )
        )
    return ticks


class TraceDroneReplayer:
    def __init__(self, ticks: List[TelemetryTick]) -> None:
        self.ticks = ticks

    def run(self, bus: TelemetryBus, tick_seconds: float = 1.0) -> None:
        """Emula o drone: a cada tick, publica telemetria no bus."""
        for tick in self.ticks:
            bus.publish(tick)
            if tick_seconds > 0:
                time.sleep(tick_seconds)

In [ ]:
# ---------- Exemplo de uso ----------
if __name__ == "__main__":
    CSV_PATH = "drone_trace_simulation.csv"
    EXECUTION_ID = 1                    # 1..4
    TICK_SECONDS = 1.0                  # 0 = sem delay

    ticks = load_ticks(CSV_PATH, execution=EXECUTION_ID)

    bus = TelemetryBus()
    monitor = RuntimeDataMonitor()
    bus.subscribe(monitor.handle_tick)

    drone = TraceDroneReplayer(ticks)
    drone.run(bus, tick_seconds=TICK_SECONDS)

    # em qualquer momento você pode consultar:
    print("\nÚltimo tick visto pelo monitor:")
    print(monitor.latest)
    print(monitor.history_runtime_data)

[exec=1 t=00] action=-               h=   0.0 b=100.0 vib=0.54
[exec=1 t=01] action=takeoff_act     h=   0.0 b=100.0 vib=0.52
[exec=1 t=02] action=checkstatus_act h= 107.2 b= 90.0 vib=0.47
[exec=1 t=03] action=flying_act      h= 104.3 b= 89.0 vib=0.54
[exec=1 t=04] action=flying_act      h= 105.9 b= 80.0 vib=0.52
[exec=1 t=05] action=landing_act     h= 105.1 b= 60.0 vib=0.54
[exec=1 t=06] action=shutdown_act    h=   0.0 b= 30.0 vib=0.53
[exec=1 t=07] action=-               h=   0.0 b= 25.0 vib=0.46

Último tick visto pelo monitor:
TelemetryTick(execution=1, t=7, h=0.0, dt=0.0, b=25.0, armed=False, wind_speed=8.4, humidity=59.1, vibration=0.46, action=None)
[TelemetryTick(execution=1, t=0, h=0.0, dt=0.0, b=100.0, armed=False, wind_speed=10.0, humidity=60.0, vibration=0.54, action=None), TelemetryTick(execution=1, t=1, h=0.0, dt=0.0, b=100.0, armed=True, wind_speed=10.1, humidity=60.5, vibration=0.52, action='takeoff_act'), TelemetryTick(execution=1, t=2, h=107.2, dt=67.0, b=90.0, armed=